# 07. Validacion de preguntas de negocio con DBC

Este notebook evalua un caso de uso distinto al `top-k` clasico: preguntas de negocio y decision preliminar para proveedores. La hipotesis es que el DBC puede aportar mas valor en tareas de interpretacion, resumen y alineacion con el rubro de la empresa que en consultas cortas de retrieval puro.

## Objetivos

- probar preguntas mas cercanas al uso real de proveedores
- comparar `base` vs `enriched` por retrieval y por contexto construido
- registrar diferencias en cobertura, pertinencia y explicabilidad
- exportar una tabla de resultados para analisis cualitativo y anexos


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from src.config import TABLES_DIR, settings
from src.rag_chain import answer_question, build_context, retrieve_documents

OUTPUT_PATH = TABLES_DIR / "business_question_validation.csv"
TOP_K = 5
GENERATE_LLM_ANSWERS = False

TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("OUTPUT_PATH:", OUTPUT_PATH)
print("TOP_K:", TOP_K)
print("GENERATE_LLM_ANSWERS:", GENERATE_LLM_ANSWERS)
print("LLM_PROVIDER:", settings.llm_provider)
print("DEFAULT_RETRIEVAL_MODE:", settings.default_retrieval_mode)


ROOT: /var/www/codigo/maestria_ia/umsa/diplomados_intermedios/dip_03
OUTPUT_PATH: /var/www/codigo/maestria_ia/umsa/diplomados_intermedios/dip_03/outputs/tables/business_question_validation.csv
TOP_K: 5
GENERATE_LLM_ANSWERS: False
LLM_PROVIDER: gemini
DEFAULT_RETRIEVAL_MODE: keyword


## 1. Definir preguntas de negocio

Estas preguntas apuntan a evaluar si el corpus enriquecido con DBC ayuda a interpretar mejor una oportunidad de contratación desde la perspectiva de un proveedor.


In [2]:
questions = [
    {
        "question_id": "biz_001",
        "category": "fit_proveedor",
        "question": "Mi empresa provee medicamentos hospitalarios. Que convocatorias recuperadas parecen mas alineadas y por que?",
    },
    {
        "question_id": "biz_002",
        "category": "fit_proveedor",
        "question": "Mi empresa vende reactivos de laboratorio. Que evidencia aparece en la convocatoria para pensar que si corresponde a nuestro rubro?",
    },
    {
        "question_id": "biz_003",
        "category": "fit_proveedor",
        "question": "Somos una empresa de software para gestion clinica. La convocatoria realmente pide software o solo equipamiento?",
    },
    {
        "question_id": "biz_004",
        "category": "resumen_requisitos",
        "question": "Resume los puntos clave que un proveedor deberia revisar antes de postular a esta convocatoria de alcantarillado.",
    },
    {
        "question_id": "biz_005",
        "category": "resumen_requisitos",
        "question": "Que insumos o materiales principales se solicitan realmente en esta convocatoria mas alla del titulo corto?",
    },
    {
        "question_id": "biz_006",
        "category": "decision_preliminar",
        "question": "Esta convocatoria parece viable para una empresa pequena o sugiere una carga documental y tecnica alta?",
    },
    {
        "question_id": "biz_007",
        "category": "decision_preliminar",
        "question": "Que senales del texto indican que esta oportunidad puede requerir requisitos formales, garantias o mayor experiencia previa?",
    },
    {
        "question_id": "biz_008",
        "category": "alineacion_rubro",
        "question": "Somos contratistas de obras sanitarias. Las convocatorias recuperadas realmente encajan con nuestro rubro o hay ruido en los resultados?",
    },
]

questions_df = pd.DataFrame(questions)
display(questions_df)


,question_id,category,question
0,biz_001,fit_proveedor,Mi empresa provee medicamentos hospitalarios. ...
1,biz_002,fit_proveedor,Mi empresa vende reactivos de laboratorio. Que...
2,biz_003,fit_proveedor,Somos una empresa de software para gestion cli...
3,biz_004,resumen_requisitos,Resume los puntos clave que un proveedor deber...
4,biz_005,resumen_requisitos,Que insumos o materiales principales se solici...
5,biz_006,decision_preliminar,Esta convocatoria parece viable para una empre...
6,biz_007,decision_preliminar,Que senales del texto indican que esta oportun...
7,biz_008,alineacion_rubro,Somos contratistas de obras sanitarias. Las co...


## 2. Ejecutar comparacion base vs enriched

Se comparan tres escenarios practicos:

- `keyword_base`
- `keyword_enriched`
- `hybrid_enriched`

La comparacion esta pensada para detectar si el DBC aporta contexto adicional util al proveedor, aunque no mejore necesariamente la metrica clasica de retrieval semantico.


In [3]:
scenarios = [
    {"scenario": "keyword_base", "retrieval_mode": "keyword", "corpus_variant": "base"},
    {"scenario": "keyword_enriched", "retrieval_mode": "keyword", "corpus_variant": "enriched"},
    {"scenario": "hybrid_enriched", "retrieval_mode": "hybrid", "corpus_variant": "enriched"},
]

rows = []

for question_row in questions:
    question = question_row["question"]
    for scenario in scenarios:
        metadata_filters = {"corpus_variant": scenario["corpus_variant"]}
        sources = retrieve_documents(
            question,
            retrieval_mode=scenario["retrieval_mode"],
            k=TOP_K,
            metadata_filters=metadata_filters,
        )
        context = build_context(
            question,
            retrieval_mode=scenario["retrieval_mode"],
            k=TOP_K,
            metadata_filters=metadata_filters,
        )

        llm_answer = ""
        if GENERATE_LLM_ANSWERS:
            response = answer_question(
                question,
                retrieval_mode=scenario["retrieval_mode"],
                k=TOP_K,
                metadata_filters=metadata_filters,
            )
            llm_answer = response.answer

        rows.append(
            {
                "question_id": question_row["question_id"],
                "category": question_row["category"],
                "scenario": scenario["scenario"],
                "retrieval_mode": scenario["retrieval_mode"],
                "corpus_variant": scenario["corpus_variant"],
                "question": question,
                "retrieved_count": len(sources),
                "retrieved_cuces": " | ".join([str(item.get("cuce", "")) for item in sources if item.get("cuce")]),
                "retrieved_entidades": " | ".join([str(item.get("entidad", "")) for item in sources if item.get("entidad")]),
                "context_chars": len(context),
                "context_preview": context[:1200],
                "llm_answer": llm_answer,
            }
        )

results_df = pd.DataFrame(rows)
results_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Resultados guardados en:", OUTPUT_PATH)
display(results_df[[
    "question_id",
    "category",
    "scenario",
    "retrieved_count",
    "retrieved_cuces",
    "context_chars",
]])


Resultados guardados en: /var/www/codigo/maestria_ia/umsa/diplomados_intermedios/dip_03/outputs/tables/business_question_validation.csv


,question_id,category,scenario,retrieved_count,retrieved_cuces,context_chars
0,biz_001,fit_proveedor,keyword_base,5,26-1101-04-1669637-1-1 | 26-634-00-1669658-1-1...,4905
1,biz_001,fit_proveedor,keyword_enriched,5,26-1276-00-1669611-1-1 | 26-1211-00-1669850-1-...,12000
2,biz_001,fit_proveedor,hybrid_enriched,5,26-1704-00-1668525-1-1 | 26-0902-43-1667332-1-...,12000
3,biz_002,fit_proveedor,keyword_base,5,26-0904-04-1667236-1-1 | 26-0904-04-1667243-1-...,4759
4,biz_002,fit_proveedor,keyword_enriched,5,26-2303-00-1668697-1-1 | 26-1701-00-1665905-1-...,12000
5,biz_002,fit_proveedor,hybrid_enriched,5,26-0902-42-1667500-1-1 | 26-0418-07-1669590-1-...,12000
6,biz_003,fit_proveedor,keyword_base,5,26-0046-38-1660991-1-1 | 26-0132-00-1664690-2-...,4759
7,biz_003,fit_proveedor,keyword_enriched,5,26-0046-38-1660991-1-1 | 26-0417-09-1669039-1-...,12000
8,biz_003,fit_proveedor,hybrid_enriched,5,26-0046-38-1660991-1-1 | 26-0035-09-1669025-1-...,12000
9,biz_004,resumen_requisitos,keyword_base,5,26-0781-00-1669540-1-1 | 26-0781-00-1669661-1-...,4821


## 3. Comparaciones rapidas por pregunta


In [4]:
for question_id in questions_df["question_id"]:
    print("=" * 100)
    print(question_id)
    subset = results_df[results_df["question_id"] == question_id].copy()
    display(subset[["scenario", "retrieved_cuces", "context_chars"]])


biz_001


,scenario,retrieved_cuces,context_chars
0,keyword_base,26-1101-04-1669637-1-1 | 26-634-00-1669658-1-1...,4905
1,keyword_enriched,26-1276-00-1669611-1-1 | 26-1211-00-1669850-1-...,12000
2,hybrid_enriched,26-1704-00-1668525-1-1 | 26-0902-43-1667332-1-...,12000


biz_002


,scenario,retrieved_cuces,context_chars
3,keyword_base,26-0904-04-1667236-1-1 | 26-0904-04-1667243-1-...,4759
4,keyword_enriched,26-2303-00-1668697-1-1 | 26-1701-00-1665905-1-...,12000
5,hybrid_enriched,26-0902-42-1667500-1-1 | 26-0418-07-1669590-1-...,12000


biz_003


...

,scenario,retrieved_cuces,context_chars
15,keyword_base,26-1201-00-1668791-1-1 | 26-0132-00-1669561-1-...,4744
16,keyword_enriched,26-1205-00-1669409-1-1 | 26-0417-05-1669845-1-...,12000
17,hybrid_enriched,26-0159-00-1669762-1-1 | 26-1101-00-1633453-3-...,12000


biz_007


,scenario,retrieved_cuces,context_chars
18,keyword_base,26-0141-00-1668280-1-1 | 26-0901-05-1669035-1-...,5176
19,keyword_enriched,26-0578-00-1669768-1-1 | 26-0291-09-1669823-1-...,12000
20,hybrid_enriched,26-0291-00-1669853-1-1 | 26-0132-00-1669693-1-...,12000


biz_008


,scenario,retrieved_cuces,context_chars
21,keyword_base,26-1201-00-1668805-1-1 | 26-1201-00-1669879-1-...,5014
22,keyword_enriched,26-2303-00-1668697-1-1 | 26-0132-00-1669693-1-...,12000
23,hybrid_enriched,26-1435-00-1669732-1-1 | 26-0417-08-1669786-1-...,12000


## 4. Revision manual focalizada

Usa esta celda para inspeccionar el contexto textual de una pregunta concreta y comparar si el corpus enriquecido aporta detalles que el corpus base no contiene.


In [5]:
QUESTION_ID_TO_INSPECT = "biz_001"

inspection_df = results_df[results_df["question_id"] == QUESTION_ID_TO_INSPECT].copy()
display(inspection_df[["scenario", "retrieved_cuces", "context_preview", "llm_answer"]])


,scenario,retrieved_cuces,context_preview,llm_answer
0,keyword_base,26-1101-04-1669637-1-1 | 26-634-00-1669658-1-1...,Documento 1\nCUCE: 26-1101-04-1669637-1-1\nEnt...,
1,keyword_enriched,26-1276-00-1669611-1-1 | 26-1211-00-1669850-1-...,Documento 1\nCUCE: 26-1276-00-1669611-1-1\nEnt...,
2,hybrid_enriched,26-1704-00-1668525-1-1 | 26-0902-43-1667332-1-...,Documento 1\nCUCE: 26-1704-00-1668525-1-1\nEnt...,


## 5. Hallazgos sugeridos

Esta seccion no calcula una metrica automatica nueva, pero te ayuda a documentar si el DBC aporta mejor explicabilidad, contexto util o soporte mas realista para preguntas de negocio.


In [6]:
summary = (
    results_df.groupby(["scenario", "category"], dropna=False)
    .agg(
        preguntas=("question_id", "nunique"),
        promedio_context_chars=("context_chars", "mean"),
        promedio_fuentes=("retrieved_count", "mean"),
    )
    .reset_index()
)
display(summary)

print("Puntos a revisar manualmente:")
print("- Si keyword_enriched aporta mas detalle util que keyword_base.")
print("- Si hybrid_enriched mejora la pertinencia o solo agrega mas texto sin foco.")
print("- Si las respuestas permiten apoyar una decision preliminar del proveedor.")
print("- Si la evidencia sirve mejor para narrativa de 'analisis documental asistido' que para 'semantic search pura'.")


,scenario,category,preguntas,promedio_context_chars,promedio_fuentes
0,hybrid_enriched,alineacion_rubro,1,12000.000000,5.0
1,hybrid_enriched,decision_preliminar,2,12000.000000,5.0
2,hybrid_enriched,fit_proveedor,3,12000.000000,5.0
3,hybrid_enriched,resumen_requisitos,2,12000.000000,5.0
4,keyword_base,alineacion_rubro,1,5014.000000,5.0
5,keyword_base,decision_preliminar,2,4960.000000,5.0
6,keyword_base,fit_proveedor,3,4807.666667,5.0
7,keyword_base,resumen_requisitos,2,4884.500000,5.0
8,keyword_enriched,alineacion_rubro,1,12000.000000,5.0
9,keyword_enriched,decision_preliminar,2,12000.000000,5.0


Puntos a revisar manualmente:
- Si keyword_enriched aporta mas detalle util que keyword_base.
- Si hybrid_enriched mejora la pertinencia o solo agrega mas texto sin foco.
- Si las respuestas permiten apoyar una decision preliminar del proveedor.
- Si la evidencia sirve mejor para narrativa de 'analisis documental asistido' que para 'semantic search pura'.
